In [1]:
import pandas as pd # import for data reading and manipulation
import re
from pathlib import Path
import numpy as np

%store -r MIMIC_DIR
%store -r cohort
%store -r demo
%store -r vaso_flag
%store -r vent_flag
%store -r vaso_events
%store -r vent_events
%store -r comorbid_flags
%store -r palliative_flag
%store -r metastatic_flag
%store -r opioids_clean
%store -r cancer_type_flags
%store -r admissions
%store -r icu_flag
%store -r naloxone_orders
%store -r sedatives

In [2]:
# LAB 
labitems = pd.read_csv("/home/emma/data/physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv") # directory 
# NOTE: unanchored "ALT"/"AST" substring matches also caught anything containing that
# substring letter-run (Casts, Yeast, Blasts, Gastrin, Thromboplastin all contain "ast"),
# pulling in ~15 unrelated, ~100%-missing columns downstream. Word boundaries + restricting
# to blood chemistry (matching how ALT/AST/Bilirubin/Creatinine are actually run) fixes it.
relevant_labs = labitems[
    (labitems.label.isin(["Creatinine", "Bilirubin, Total"]) |
     labitems.label.str.contains(r"\bALT\b|alanine aminotransferase|\bSGPT\b", case=False, na=False, regex=True) |
     labitems.label.str.contains(r"\bAST\b|aspartate aminotransferase|\bSGOT\b", case=False, na=False, regex=True))
    & (labitems.fluid == "Blood")
]
lab_itemids = set(relevant_labs.itemid.unique())

print("Labels found: ", relevant_labs.label.unique())


Labels found:  <ArrowStringArray>
[ 'Alanine Aminotransferase (ALT)', 'Asparate Aminotransferase (AST)',
                'Bilirubin, Total',                      'Creatinine',
        'Alanine Aminotransferase']
Length: 5, dtype: str


In [3]:
cancer_cohort = pd.read_parquet("cancer_cohort.parquet")
cancer_cohort_hadm_ids = set(cancer_cohort.hadm_id.unique())
chunks = []


reader = pd.read_csv(MIMIC_DIR / "labevents.csv", chunksize=1_000_000, # reads to the lab events csv
                    usecols = ["subject_id", "hadm_id", "itemid", "valuenum", "charttime"]
)

for chunk in reader: 
    chunk = chunk[chunk.hadm_id.isin(cancer_cohort.hadm_id) & chunk.itemid.isin(lab_itemids)]
    if len(chunk):
        chunks.append(chunk)
    
labs = pd.concat(chunks, ignore_index=True)
labs = labs.merge(relevant_labs[["itemid", "label"]], on="itemid")

lab_summary = (
    labs.groupby(["hadm_id", "label"])["valuenum"]
    .agg(["mean", "max"])
    .unstack("label")
)

lab_summary.columns = ["_".join(col).strip() for col in lab_summary.columns.values]
lab_summary = lab_summary.reset_index()

print(lab_summary.columns.tolist())
lab_summary.to_parquet("lab_summary.parquet") # saves to parquet

%store lab_summary

['hadm_id', 'mean_Alanine Aminotransferase (ALT)', 'mean_Asparate Aminotransferase (AST)', 'mean_Bilirubin, Total', 'mean_Creatinine', 'max_Alanine Aminotransferase (ALT)', 'max_Asparate Aminotransferase (AST)', 'max_Bilirubin, Total', 'max_Creatinine']
Stored 'lab_summary' (DataFrame)


In [4]:
icustays = pd.read_csv(
    "/home/emma/data/physionet.org/files/mimiciv/3.1/icu/icustays.csv",
    usecols = ["subject_id", "hadm_id", "stay_id", "intime", "outtime", "los"]
)
d_items = pd.read_csv("/home/emma/data/physionet.org/files/mimiciv/3.1/icu/d_items.csv")
d_labitems = pd.read_csv(MIMIC_DIR / "d_labitems.csv")

CHART_ITEMIDS = {
    "spo2": 220277,
    "fio2": 223835,
    "map": 220052,
    "map_cuff": 220181,
    "gcs_eye": 220739,
    "gcs_verbal": 223900,
    "gcs_motor": 223901,
    "resp_rate": 220210,
}

print(d_labitems[d_labitems.label.str.contains("platelet", case = False)])
print(d_labitems[d_labitems.label.str.contains("bilirubin", case = False)])
print(d_labitems[d_labitems.label.str.contains("creatinine", case = False)])

LAB_ITEMIDS = {
    "platelets": 51265,
    "bilirubin": 50885,
    "creatinine": 50912,
}


      itemid                           label  fluid    category
425    51240                 Large Platelets  Blood  Hematology
449    51264                 Platelet Clumps  Blood  Hematology
450    51265                  Platelet Count  Blood  Hematology
451    51266                  Platelet Smear  Blood  Hematology
1195   52105  Direct Antiplatelet Antibodies  Blood  Hematology
1231   52142            Mean Platelet Volume  Blood  Hematology
1248   52159            Platelet Aggregation  Blood  Hematology
1648   53189                  Platelet Count  Blood   Chemistry
      itemid                          label                fluid    category
36     50838      Bilirubin, Total, Ascites              Ascites   Chemistry
81     50883              Bilirubin, Direct                Blood   Chemistry
82     50884            Bilirubin, Indirect                Blood   Chemistry
83     50885               Bilirubin, Total                Blood   Chemistry
216    51028   Bilirubin, Total, Body F

In [5]:
icu_hadm_ids = set(icustays[icustays.hadm_id.isin(cohort.hadm_id)].hadm_id)
chart_itemids_needed = set(CHART_ITEMIDS.values())

chart_chunks = []
reader = pd.read_csv(
    "/home/emma/data/physionet.org/files/mimiciv/3.1/icu/chartevents.csv",
    usecols = ["subject_id", "hadm_id", "stay_id", "itemid", "charttime", "valuenum"],
    chunksize = 5_000_000,
    dtype = {"subject_id": "int64", "hadm_id": "Int64"},
)

for i, chunk in enumerate(reader):
    chunk = chunk[
        chunk.hadm_id.isin(icu_hadm_ids) &
        chunk.itemid.isin(chart_itemids_needed) &
        chunk.valuenum.notna()
    ]
    if len(chunk):
        chart_chunks.append(chunk)
    if i % 10 == 0:
        print(f"Processed {i} chunks, rows: {sum(len(c) for c in chart_chunks):,}")
chartevents = pd.concat(chart_chunks, ignore_index=True)
chartevents["charttime"] = pd.to_datetime(chartevents["charttime"])
print(f"chartevents pulled: {len(chartevents):,} rows")

Processed 0 chunks, rows: 65,895
Processed 10 chunks, rows: 615,133
Processed 20 chunks, rows: 1,172,815
Processed 30 chunks, rows: 1,760,358
Processed 40 chunks, rows: 2,341,238
Processed 50 chunks, rows: 2,852,218
Processed 60 chunks, rows: 3,411,406
Processed 70 chunks, rows: 3,969,006
Processed 80 chunks, rows: 4,527,324
chartevents pulled: 4,803,797 rows


In [6]:
lab_itemids_needed = set(LAB_ITEMIDS.values())
lab_chunks = []

reader = pd.read_csv(
    MIMIC_DIR / "labevents.csv",
    usecols = ["subject_id", "hadm_id", "itemid", "charttime", "valuenum"],
    chunksize = 5_000_000,
    dtype = {"subject_id": "int64", "hadm_id": "Int64"},
)

# NOTE: labevents.csv is hospital-wide (unlike chartevents, which is ICU-only), so this
# pulls labs for the whole cohort, not just ICU-associated admissions -- that's what lets
# the pre-dose coagulation/liver/renal signals below reach non-ICU doses too.
for chunk in reader:
    chunk = chunk[
        chunk.hadm_id.isin(cohort.hadm_id) &
        chunk.itemid.isin(lab_itemids_needed) &
        chunk.valuenum.notna()
    ]
    if len(chunk):
        lab_chunks.append(chunk)
labevents_sofa = pd.concat(lab_chunks, ignore_index=True)
labevents_sofa["charttime"] = pd.to_datetime(labevents_sofa["charttime"])

In [7]:
output_chunks = []
reader = pd.read_csv(
    "/home/emma/data/physionet.org/files/mimiciv/3.1/icu/outputevents.csv",
    usecols = ["subject_id", "hadm_id", "stay_id", "itemid", "charttime", "value"],
    chunksize = 1_000_000 

)

urine_itemids = set(
    d_items[d_items.label.str.contains("urine|foley", case = False, na = False)]["itemid"]

)
for chunk in reader:
    chunk = chunk[
        chunk.hadm_id.isin(icu_hadm_ids) &
        chunk.itemid.isin(urine_itemids)
    ]
    if len(chunk):
        output_chunks.append(chunk)

urine = pd.concat(output_chunks, ignore_index=True)
urine["charttime"] = pd.to_datetime(urine["charttime"])
urine["value"] = pd.to_numeric(urine["value"], errors = "coerce")

In [8]:
icustays["intime"] = pd.to_datetime(icustays["intime"])
icustays["outtime"] = pd.to_datetime(icustays["outtime"])

def worst_per_stay(events_df, itemid, agg="min", label="value"):
    """
    for each itemid, find the worse value in an ICU window
    min = low = bad (platelets, MAP, GCS)
    max = high = bad (bilirubin, creatinine)
    """
    sub = events_df[events_df.itemid == itemid][
        ["hadm_id", "stay_id", "charttime", "valuenum"]
    ].copy()

    sub = sub.merge(
        icustays[["stay_id", "hadm_id", "intime", "outtime"]],
        on = ["stay_id", "hadm_id"], how = "left"
    )

    sub = sub[(sub.charttime >= sub.intime) & (sub.charttime <= sub.outtime)]

    result = (
        sub.groupby("hadm_id")["valuenum"]
        .agg(agg)
        .reset_index()
        .rename(columns = {"valuenum": label})
    )
    return result

spo2_worst = worst_per_stay(chartevents, CHART_ITEMIDS["spo2"], agg = "min", label = "spo2_min")
fio2_worst = worst_per_stay(chartevents, CHART_ITEMIDS["fio2"], agg = "max", label = "fio2_max")

map_art = worst_per_stay(chartevents, CHART_ITEMIDS["map"], agg = "min", label = "map_min")
map_cuff = worst_per_stay(chartevents, CHART_ITEMIDS["map_cuff"], agg = "min", label = "map_cuff_min")

gcs_eye = worst_per_stay(chartevents, CHART_ITEMIDS["gcs_eye"], agg = "min", label = "gcs_eye_min")
gcs_verbal = worst_per_stay(chartevents, CHART_ITEMIDS["gcs_verbal"], agg = "min", label = "gcs_verbal_min")
gcs_motor = worst_per_stay(chartevents, CHART_ITEMIDS["gcs_motor"], agg = "min", label = "gcs_motor_min")

def worst_lab(itemid, agg, label):
    sub = labevents_sofa[labevents_sofa.itemid ==itemid][
        ["hadm_id", "charttime", "valuenum"]
    ].copy()
    icu_windows = (
        icustays.groupby("hadm_id")
        .agg(intime = ("intime", "min"), outtime = ("outtime", "max"))
        .reset_index() 
    )
    sub = sub.merge(icu_windows, on = "hadm_id", how = "left")
    
    sub = sub[(sub.charttime >= sub.intime) & (sub.charttime <= sub.outtime)]
    return(
        sub.groupby("hadm_id")["valuenum"]
        .agg(agg)
        .reset_index()
        .rename(columns = {"valuenum": label})
    )
platelets_worst = worst_lab(LAB_ITEMIDS["platelets"], "min", "platelets_min")
bilirubin_worst = worst_lab(LAB_ITEMIDS["bilirubin"], "max", "bilirubin_max")
creatinine_worst = worst_lab(LAB_ITEMIDS["creatinine"], "max", "creatinine_max_icu")
urine_merged = (
    urine.merge(
        icustays[["stay_id", "hadm_id", "intime", "outtime", "los"]],
        on = ["stay_id", "hadm_id"], how = "left"
    )
    .query("charttime >= intime and charttime <= outtime")
)
urine_summary = (
    urine_merged.groupby("hadm_id")
    .agg(total_urine_ml = ("value", "sum"), icu_los_days = ("los", "first"))
    .reset_index()
)
urine_summary["urine_ml_per_day"] = (
    urine_summary["total_urine_ml"] / urine_summary["icu_los_days"]
)


In [9]:
sofa = (
    icustays[["hadm_id"]].drop_duplicates()
    .merge(spo2_worst,       on="hadm_id", how="left")
    .merge(fio2_worst,       on="hadm_id", how="left")
    .merge(map_art,          on="hadm_id", how="left")
    .merge(map_cuff,         on="hadm_id", how="left")
    .merge(gcs_eye,          on="hadm_id", how="left")
    .merge(gcs_verbal,       on="hadm_id", how="left")
    .merge(gcs_motor,        on="hadm_id", how="left")
    .merge(platelets_worst,  on="hadm_id", how="left")
    .merge(bilirubin_worst,  on="hadm_id", how="left")
    .merge(creatinine_worst, on="hadm_id", how="left")
    .merge(urine_summary[["hadm_id", "urine_ml_per_day"]], on="hadm_id", how="left")
    .merge(vaso_flag[["hadm_id", "received_vasopressors"]], on="hadm_id", how="left")
    .merge(vent_flag[["hadm_id", "mechanically_ventilated"]], on="hadm_id", how="left")
)

sofa["map_worst"] = sofa["map_min"].fillna(sofa["map_cuff_min"])

sofa["gcs_total"] = (
    sofa[["gcs_eye_min", "gcs_verbal_min", "gcs_motor_min"]].sum(axis=1)

)
sofa["gcs_total"] = sofa["gcs_total"].where(sofa["gcs_total"] > 0, np.nan)

# NOTE: itemid 223835 (FiO2) is charted 21-100 (percent) in MIMIC-IV, not as a 0-1 fraction
sofa["fio2_fraction"] = sofa["fio2_max"] / 100
sofa["spo2_fio2_ratio"] = sofa["spo2_min"] / sofa["fio2_fraction"]

def score_respiration(row):
    ratio = row["spo2_fio2_ratio"]
    vented = row.get("mechanically_ventilated", False)
    if pd.isna(ratio):   return np.nan
    if ratio < 67:       return 4 if vented else 3
    if ratio < 142:      return 3 if vented else 2
    if ratio < 221:      return 2
    if ratio < 302:      return 1
    return 0

def score_coagulation(platelets):
    if pd.isna(platelets): return np.nan
    if platelets < 20:  return 4
    if platelets < 50:  return 3
    if platelets < 100: return 2
    if platelets < 150: return 1
    return 0

def score_liver(bilirubin):
    if pd.isna(bilirubin): return np.nan
    if bilirubin >= 12.0: return 4
    if bilirubin >= 6.0:  return 3
    if bilirubin >= 2.0:  return 2
    if bilirubin >= 1.2:  return 1
    return 0

def score_cardiovascular(row):
    map_val = row["map_worst"]
    on_vasopressors = row.get("received_vasopressors", False)
    if pd.isna(map_val) and not on_vasopressors: return np.nan
    if on_vasopressors: return 3  # simplified — full SOFA uses specific drug doses
    if map_val < 70:    return 1
    return 0

def score_cns(gcs):
    if pd.isna(gcs):  return np.nan
    if gcs < 6:   return 4
    if gcs < 10:  return 3
    if gcs < 13:  return 2
    if gcs < 15:  return 1
    return 0

def score_renal(row):
    cr = row["creatinine_max_icu"]
    uo = row["urine_ml_per_day"]
    score = 0
    if pd.notna(cr):
        if cr >= 5.0:   score = max(score, 4)
        elif cr >= 3.5: score = max(score, 3)
        elif cr >= 2.0: score = max(score, 2)
        elif cr >= 1.2: score = max(score, 1)
    if pd.notna(uo):
        if uo < 200:    score = max(score, 4)
        elif uo < 500:  score = max(score, 3)
    return score 
sofa["sofa_respiration"]    = sofa.apply(score_respiration, axis=1)
sofa["sofa_coagulation"]    = sofa["platelets_min"].apply(score_coagulation)
sofa["sofa_liver"]          = sofa["bilirubin_max"].apply(score_liver)
sofa["sofa_cardiovascular"] = sofa.apply(score_cardiovascular, axis=1)
sofa["sofa_cns"]            = sofa["gcs_total"].apply(score_cns)
sofa["sofa_renal"]          = sofa.apply(score_renal, axis=1)

sofa_component_cols = [
    "sofa_respiration", "sofa_coagulation", "sofa_liver",
    "sofa_cardiovascular", "sofa_cns", "sofa_renal"
]

sofa["sofa_total"] = sofa[sofa_component_cols].sum(axis=1, min_count=1)
sofa["n_sofa_components_missing"] = sofa[sofa_component_cols].isna().sum(axis=1)

print(sofa["sofa_total"].describe())
print(f"\nPatients with ≥3 missing components: {(sofa.n_sofa_components_missing >= 3).sum()}")

%store sofa
%store sofa_component_cols

count    85242.000000
mean         3.786924
std          2.460030
min          3.000000
25%          3.000000
50%          3.000000
75%          3.000000
max         23.000000
Name: sofa_total, dtype: float64

Patients with ≥3 missing components: 72740
Stored 'sofa' (DataFrame)
Stored 'sofa_component_cols' (list)


In [10]:
# ── DOSE-LEVEL TABLE: pre-dose severity snapshot ─────────────────────────────
# Vitals/GCS (chartevents) only exist for ICU stays, so the respiration/cardiovascular/CNS
# pre-dose signals below are ICU-only. Labs (labevents_sofa) are hospital-wide, so the
# coagulation/liver/renal pre-dose signals are available for non-ICU doses too.
opioids_clean["starttime"] = pd.to_datetime(opioids_clean["starttime"])

icu_hadm_set = set(icustays.hadm_id.unique())

# NOTE: a handful of opioids_clean rows (57 of 488,455, 53 within this cohort) have a
# null starttime -- merge_asof requires a non-null, sortable "on" key, and a dose with no
# known start time can't be placed in the pre-dose lookback windows or dosing-order
# calcs below anyway, so those rows are dropped here rather than downstream.
doses = (
    opioids_clean[opioids_clean.hadm_id.isin(cohort.hadm_id)]
    .dropna(subset=["starttime"])
    .reset_index(drop=True)
    .reset_index()
    .rename(columns={"index": "dose_id"})
)
doses["in_icu_hadm"] = doses["hadm_id"].isin(icu_hadm_set)

def nearest_prior_value(events_df, itemid, doses_df, out_col, window_hours=24):
    """
    For each dose, the most recent valuenum charted for `itemid` within
    `window_hours` before the dose's starttime (per hadm_id). Vectorized via
    merge_asof instead of a per-row loop over doses.
    """
    left = doses_df[["dose_id", "hadm_id", "starttime"]].sort_values("starttime")
    right = (
        events_df[events_df.itemid == itemid][["hadm_id", "charttime", "valuenum"]]
        .dropna(subset=["valuenum"])
        .sort_values("charttime")
    )
    merged = pd.merge_asof(
        left, right,
        left_on="starttime", right_on="charttime",
        by="hadm_id", direction="backward",
        tolerance=pd.Timedelta(hours=window_hours),
    )
    return merged.set_index("dose_id")["valuenum"].rename(out_col)

# vitals/GCS: 24h lookback (frequently charted in the ICU)
pre_dose_signals = [
    nearest_prior_value(chartevents, CHART_ITEMIDS["spo2"],       doses, "spo2_pre_dose"),
    nearest_prior_value(chartevents, CHART_ITEMIDS["fio2"],       doses, "fio2_pre_dose"),
    nearest_prior_value(chartevents, CHART_ITEMIDS["map"],        doses, "map_pre_dose"),
    nearest_prior_value(chartevents, CHART_ITEMIDS["map_cuff"],   doses, "map_cuff_pre_dose"),
    nearest_prior_value(chartevents, CHART_ITEMIDS["gcs_eye"],    doses, "gcs_eye_pre_dose"),
    nearest_prior_value(chartevents, CHART_ITEMIDS["gcs_verbal"], doses, "gcs_verbal_pre_dose"),
    nearest_prior_value(chartevents, CHART_ITEMIDS["gcs_motor"],  doses, "gcs_motor_pre_dose"),
    # labs: 48h lookback (often drawn once daily, not every few hours like vitals)
    nearest_prior_value(labevents_sofa, LAB_ITEMIDS["platelets"],  doses, "platelets_pre_dose",  window_hours=48),
    nearest_prior_value(labevents_sofa, LAB_ITEMIDS["bilirubin"],  doses, "bilirubin_pre_dose",  window_hours=48),
    nearest_prior_value(labevents_sofa, LAB_ITEMIDS["creatinine"], doses, "creatinine_pre_dose", window_hours=48),
]

doses = doses.set_index("dose_id")
for s in pre_dose_signals:
    doses = doses.join(s)
doses = doses.reset_index()

doses["map_pre_dose_worst"] = doses["map_pre_dose"].fillna(doses["map_cuff_pre_dose"])
doses["gcs_total_pre_dose"] = doses[["gcs_eye_pre_dose", "gcs_verbal_pre_dose", "gcs_motor_pre_dose"]].sum(axis=1)
doses["gcs_total_pre_dose"] = doses["gcs_total_pre_dose"].where(doses["gcs_total_pre_dose"] > 0, np.nan)

# NOTE: itemid 223835 (FiO2) is charted 21-100 (percent) in MIMIC-IV, not as a 0-1 fraction
doses["fio2_fraction_pre_dose"] = doses["fio2_pre_dose"] / 100
doses["spo2_fio2_ratio_pre_dose"] = doses["spo2_pre_dose"] / doses["fio2_fraction_pre_dose"]

print(f"Doses: {len(doses):,}  |  in an ICU-associated admission: {doses['in_icu_hadm'].sum():,}")
print(f"Doses with a pre-dose SpO2/FiO2 reading: {doses['spo2_fio2_ratio_pre_dose'].notna().sum():,}")

Doses: 153,232  |  in an ICU-associated admission: 40,822
Doses with a pre-dose SpO2/FiO2 reading: 8,257


In [11]:
# ── DOSE-LEVEL TABLE: pre-dose SOFA scores (reusing the admission-level scoring rules) ──
# NOTE: vaso_flag/vent_flag are whole-admission "ever happened" flags. Using them directly
# here would leak forward in time -- an early dose could get scored as if the patient were
# already on vasopressors/ventilated when those actually started later in the stay. Instead,
# check whether a vasopressor/ventilation interval was already active at each dose's
# starttime, using the raw timed events (same interval-overlap approach as
# sedative_active_at_dose below).
def interval_active_at_times(dose_times, ev_rows):
    starts = ev_rows["starttime"].to_numpy()
    ends = ev_rows["endtime"].fillna(ev_rows["starttime"]).to_numpy()
    return [bool(((starts <= t) & (ends >= t)).any()) for t in dose_times]

vent_events_valid = vent_events.dropna(subset=["starttime", "hadm_id"])
vaso_events_valid = vaso_events.dropna(subset=["starttime", "hadm_id"])
vent_by_hadm = dict(tuple(vent_events_valid.groupby("hadm_id")))
vaso_by_hadm = dict(tuple(vaso_events_valid.groupby("hadm_id")))

doses["mechanically_ventilated_at_dose"] = False
doses["received_vasopressors_at_dose"] = False

for hadm_id, dose_rows in doses.groupby("hadm_id"):
    dose_times = dose_rows["starttime"].to_numpy()
    vent_rows = vent_by_hadm.get(hadm_id)
    if vent_rows is not None and not vent_rows.empty:
        doses.loc[dose_rows.index, "mechanically_ventilated_at_dose"] = interval_active_at_times(dose_times, vent_rows)
    vaso_rows = vaso_by_hadm.get(hadm_id)
    if vaso_rows is not None and not vaso_rows.empty:
        doses.loc[dose_rows.index, "received_vasopressors_at_dose"] = interval_active_at_times(dose_times, vaso_rows)

print(f"Doses with mechanical ventilation active at dose time: {doses['mechanically_ventilated_at_dose'].mean():.1%}")
print(f"Doses with vasopressors active at dose time:           {doses['received_vasopressors_at_dose'].mean():.1%}")

# urine output isn't windowed per-dose in this pass (rate-based, not an instantaneous
# reading) — renal scoring below falls back to creatinine-only, same as the functions
# already handle when urine is missing.
score_input = pd.DataFrame({
    "spo2_fio2_ratio":         doses["spo2_fio2_ratio_pre_dose"],
    "map_worst":               doses["map_pre_dose_worst"],
    "gcs_total":               doses["gcs_total_pre_dose"],
    "platelets_min":           doses["platelets_pre_dose"],
    "bilirubin_max":           doses["bilirubin_pre_dose"],
    "creatinine_max_icu":      doses["creatinine_pre_dose"],
    "urine_ml_per_day":        np.nan,
    "mechanically_ventilated": doses["mechanically_ventilated_at_dose"],
    "received_vasopressors":   doses["received_vasopressors_at_dose"],
})

doses["sofa_respiration_pre_dose"]    = score_input.apply(score_respiration, axis=1)
doses["sofa_coagulation_pre_dose"]    = score_input["platelets_min"].apply(score_coagulation)
doses["sofa_liver_pre_dose"]          = score_input["bilirubin_max"].apply(score_liver)
doses["sofa_cardiovascular_pre_dose"] = score_input.apply(score_cardiovascular, axis=1)
doses["sofa_cns_pre_dose"]            = score_input["gcs_total"].apply(score_cns)
doses["sofa_renal_pre_dose"]          = score_input.apply(score_renal, axis=1)

pre_dose_sofa_cols = [
    "sofa_respiration_pre_dose", "sofa_coagulation_pre_dose", "sofa_liver_pre_dose",
    "sofa_cardiovascular_pre_dose", "sofa_cns_pre_dose", "sofa_renal_pre_dose",
]
doses["sofa_total_pre_dose"] = doses[pre_dose_sofa_cols].sum(axis=1, min_count=1)
doses["n_sofa_components_missing_pre_dose"] = doses[pre_dose_sofa_cols].isna().sum(axis=1)

print(doses["sofa_total_pre_dose"].describe())


Doses with mechanical ventilation active at dose time: 1.8%
Doses with vasopressors active at dose time:           1.3%
count    153232.000000
mean          1.160867
std           1.907594
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max          20.000000
Name: sofa_total_pre_dose, dtype: float64


In [12]:
# ── DOSE-LEVEL TABLE: mortality outcome, multiple horizons ───────────────────
# Several horizons let you check whether the dose-mortality association gets stronger at
# short latency (more consistent with an acute pharmacologic effect) or stays flat across
# horizons (more consistent with confounding by severity/comfort care).
#
# deathtime is a precise in-hospital timestamp; dod (patients.csv) is date-only with no
# time-of-day, which can misclassify a death by up to ~24h at these short horizons. Prefer
# deathtime (covers essentially all in-hospital deaths) and only fall back to dod for deaths
# recorded after discharge, where no timestamp exists.
death_time_lookup = pd.to_datetime(cohort.set_index("hadm_id")["deathtime"])
dod_lookup = pd.to_datetime(cohort.set_index("hadm_id")["dod"])

doses["deathtime"] = doses["hadm_id"].map(death_time_lookup)
doses["dod"] = doses["hadm_id"].map(dod_lookup)
doses["death_reference_time"] = doses["deathtime"].fillna(doses["dod"])

doses["hours_to_death"] = (doses["death_reference_time"] - doses["starttime"]).dt.total_seconds() / 3600

for horizon in [6, 12, 24, 48]:
    doses[f"died_within_{horizon}h"] = doses["hours_to_death"].between(0, horizon) & doses["death_reference_time"].notna()

print(f"Doses: {len(doses):,}")
n_precise = doses["deathtime"].notna().sum()
n_date_only = (doses["deathtime"].isna() & doses["dod"].notna()).sum()
print(f"Deaths with a precise deathtime: {n_precise:,}  |  date-only (dod) fallback: {n_date_only:,}")
for horizon in [6, 12, 24, 48]:
    col = f"died_within_{horizon}h"
    print(f"Doses followed by death within {horizon}h: {doses[col].sum():,} ({doses[col].mean():.2%})")

Doses: 153,232
Deaths with a precise deathtime: 16,704  |  date-only (dod) fallback: 86,748
Doses followed by death within 6h: 1,288 (0.84%)
Doses followed by death within 12h: 2,134 (1.39%)
Doses followed by death within 24h: 3,498 (2.28%)
Doses followed by death within 48h: 5,572 (3.64%)


In [13]:
# ── DOSE-LEVEL TABLE: sedative co-administration + opioid dosing escalation ──
# Both are computed relative to *this specific dose's* starttime only, using orders
# up to and including that moment, so neither leaks information from later in the stay.
sedatives = sedatives.copy()
sedatives["starttime"] = pd.to_datetime(sedatives["starttime"])
sedatives["stoptime"] = pd.to_datetime(sedatives["stoptime"])
sedatives_valid = sedatives.dropna(subset=["starttime", "stoptime", "hadm_id"])

def sedative_features_for_hadm(dose_times, sed_rows):
    starts = sed_rows["starttime"].to_numpy()
    stops = sed_rows["stoptime"].to_numpy()
    active = [bool(((starts <= t) & (stops >= t)).any()) for t in dose_times]
    prior = [int((starts < t).sum()) for t in dose_times]
    return active, prior

doses["sedative_active_at_dose"] = False
doses["prior_sedative_orders"] = 0

sed_by_hadm = dict(tuple(sedatives_valid.groupby("hadm_id")))
for hadm_id, dose_rows in doses.groupby("hadm_id"):
    sed_rows = sed_by_hadm.get(hadm_id)
    if sed_rows is None or sed_rows.empty:
        continue
    active, prior = sedative_features_for_hadm(dose_rows["starttime"].to_numpy(), sed_rows)
    doses.loc[dose_rows.index, "sedative_active_at_dose"] = active
    doses.loc[dose_rows.index, "prior_sedative_orders"] = prior

print(f"Doses with a sedative active at time of opioid dose: {doses['sedative_active_at_dose'].mean():.1%}")

# ── Opioid dosing escalation: MME burden already administered earlier in *this* admission ──
doses = doses.sort_values(["hadm_id", "starttime"]).copy()
doses["cumulative_mme_prior_doses"] = (
    doses.groupby("hadm_id")["mme_dose"].cumsum() - doses["mme_dose"]
)
doses["n_prior_opioid_doses_this_admission"] = doses.groupby("hadm_id").cumcount()

print(doses[["cumulative_mme_prior_doses", "n_prior_opioid_doses_this_admission"]].describe())

Doses with a sedative active at time of opioid dose: 57.8%
       cumulative_mme_prior_doses  n_prior_opioid_doses_this_admission
count               153232.000000                        153232.000000
mean                  1603.221227                             4.215673
std                   6617.169069                             7.097450
min                      0.000000                             0.000000
25%                      0.000000                             0.000000
50%                     17.000000                             2.000000
75%                    160.000000                             5.000000
max                 251677.500000                           117.000000


In [14]:
# ── DOSE-LEVEL TABLE: opioid-toxicity adverse-event outcome (naloxone) ───────
# Naloxone given shortly *after* this specific dose is a much cleaner adverse-event signal
# than the whole-admission flag: it's windowed forward only, so it can't leak information
# from later in the stay, and it's specific to this dose rather than "at some point during
# this hospitalization." Treat these as secondary outcomes (opioid-related toxicity),
# not predictors of died_within_*h.
naloxone_orders = naloxone_orders.copy()
naloxone_orders["starttime"] = pd.to_datetime(naloxone_orders["starttime"])

def next_event_within(events_df, doses_df, out_col, window_hours):
    left = doses_df[["dose_id", "hadm_id", "starttime"]].sort_values("starttime")
    right = (
        events_df[["hadm_id", "starttime"]]
        .rename(columns={"starttime": "event_time"})
        .dropna(subset=["event_time"])
        .sort_values("event_time")
    )
    merged = pd.merge_asof(
        left, right,
        left_on="starttime", right_on="event_time",
        by="hadm_id", direction="forward",
        tolerance=pd.Timedelta(hours=window_hours),
    )
    return merged.set_index("dose_id")["event_time"].notna().rename(out_col)

doses = doses.set_index("dose_id")
for window_hours in [6, 12]:
    signal = next_event_within(naloxone_orders, doses.reset_index(), f"naloxone_within_{window_hours}h", window_hours)
    doses = doses.join(signal)
doses = doses.reset_index()

print(f"Doses followed by naloxone within 6h:  {doses['naloxone_within_6h'].sum():,} ({doses['naloxone_within_6h'].mean():.2%})")
print(f"Doses followed by naloxone within 12h: {doses['naloxone_within_12h'].sum():,} ({doses['naloxone_within_12h'].mean():.2%})")

Doses followed by naloxone within 6h:  2,082 (1.36%)
Doses followed by naloxone within 12h: 2,253 (1.47%)


In [15]:
# ── DOSE-LEVEL TABLE: assemble modeling-ready covariates ─────────────────────
dose_level_df = (
    doses
    .merge(demo[["subject_id", "hadm_id", "age_at_admission"]], on=["subject_id", "hadm_id"], how="left")
    .merge(cohort[["hadm_id", "anchor_age", "gender", "exposed"]], on="hadm_id", how="left")
    .merge(comorbid_flags, on="hadm_id", how="left")
    .merge(palliative_flag[["hadm_id", "has_palliative_icd", "any_palliative_signal"]], on="hadm_id", how="left")
    .merge(metastatic_flag, on="hadm_id", how="left")
    .merge(cancer_type_flags, on="hadm_id", how="left")
    .merge(admissions[["hadm_id", "insurance", "race", "language"]], on="hadm_id", how="left")
    .merge(icu_flag[["hadm_id", "n_icu_stays", "had_icu_stay"]], on="hadm_id", how="left")
)

bool_cols = ["has_palliative_icd", "any_palliative_signal", "has_metastatic_disease", "had_icu_stay"]
dose_level_df[bool_cols] = dose_level_df[bool_cols].fillna(False)

cancer_type_cols = [c for c in cancer_type_flags.columns if c.startswith("cancer_")]
dose_level_df[cancer_type_cols] = dose_level_df[cancer_type_cols].fillna(False)
dose_level_df["n_cancer_types_flagged"] = dose_level_df["n_cancer_types_flagged"].fillna(0)

dose_level_df["n_icu_stays"] = dose_level_df["n_icu_stays"].fillna(0)
for col in ["insurance", "race", "language"]:
    dose_level_df[col] = dose_level_df[col].fillna("Unknown")

# NOTE ON LEAKAGE: `total_icu_los_days` was deliberately left out of this table -- it's a
# whole-admission total that can include ICU time occurring after a given dose. Whole-admission
# naloxone use (`naloxone_flag`) was dropped entirely as a predictor for the same reason; use
# the per-dose, forward-windowed `naloxone_within_6h` / `naloxone_within_12h` columns as
# adverse-event outcomes instead (see the naloxone cell above), not as covariates for
# died_within_*h.

print(f"Final dose-level table: {dose_level_df.shape}")
print(f"Unique admissions represented: {dose_level_df.hadm_id.nunique():,}")
for horizon in [6, 12, 24, 48]:
    col = f"died_within_{horizon}h"
    print(f"{col} rate overall: {dose_level_df[col].mean():.2%}")
print(f"\ndied_within_48h rate, ICU-associated: {dose_level_df.loc[dose_level_df.in_icu_hadm, 'died_within_48h'].mean():.2%}")
print(f"died_within_48h rate, non-ICU:        {dose_level_df.loc[~dose_level_df.in_icu_hadm, 'died_within_48h'].mean():.2%}")
print(f"\nComfort-care signal present (any_palliative_signal): {dose_level_df.any_palliative_signal.mean():.2%}")
print(f"Opioid-naive doses (patient's first recorded opioid order): {dose_level_df.opioid_naive.mean():.1%}")
print(f"Doses with a sedative active concurrently: {dose_level_df.sedative_active_at_dose.mean():.1%}")
print(f"Doses followed by naloxone within 6h: {dose_level_df.naloxone_within_6h.mean():.2%}")

dose_level_df.to_parquet("dose_level_df.parquet")
%store dose_level_df

Final dose-level table: (153232, 85)
Unique admissions represented: 43,914
died_within_6h rate overall: 0.84%
died_within_12h rate overall: 1.39%
died_within_24h rate overall: 2.28%
died_within_48h rate overall: 3.64%

died_within_48h rate, ICU-associated: 8.23%
died_within_48h rate, non-ICU:        1.97%

Comfort-care signal present (any_palliative_signal): 19.07%
Opioid-naive doses (patient's first recorded opioid order): 14.4%
Doses with a sedative active concurrently: 57.8%
Doses followed by naloxone within 6h: 1.36%
Stored 'dose_level_df' (DataFrame)


In [16]:
# ── AGE FILTER: restrict to adult patients (18+) ─────────────────────────────
print(f"Before age filter: {len(dose_level_df):,} doses, {dose_level_df.hadm_id.nunique():,} admissions")
dose_level_df = dose_level_df[dose_level_df["anchor_age"] >= 18].copy().reset_index(drop=True)
print(f"After age filter:  {len(dose_level_df):,} doses, {dose_level_df.hadm_id.nunique():,} admissions")
print(f"Any under 18: {(dose_level_df.anchor_age < 18).sum()}")  # should be 0

dose_level_df.to_parquet("dose_level_df.parquet")
%store dose_level_df

Before age filter: 153,232 doses, 43,914 admissions
After age filter:  153,232 doses, 43,914 admissions
Any under 18: 0
Stored 'dose_level_df' (DataFrame)


In [17]:
dose_level_df = pd.read_parquet("dose_level_df.parquet")
dose_level_df.to_csv("dose_level_df.csv", index = False)
print(f"Saved, Shape: {dose_level_df.shape}")

Saved, Shape: (153232, 85)
